In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Define the path of your folder containing 60 zips
zip_folder = Path(r"C:\Users\ASUS\Desktop\tennis_data")

# Read proper tables
PeriodInfo_df = pd.read_parquet(zip_folder / "statistics_all.parquet")

In [2]:
# Now, we only focus on aces, so we filter our table and drop duplicates
PeriodInfo_df = PeriodInfo_df[(PeriodInfo_df["statistic_name"]=="aces") &
                              (PeriodInfo_df["period"]=="ALL")]
PeriodInfo_df.drop(columns=["date"],inplace=True)
PeriodInfo_df.drop_duplicates(inplace=True)

# Convert number of aces by each side to enable calculating total aces
PeriodInfo_df["home_stat"] = PeriodInfo_df["home_stat"].astype("Float64")
PeriodInfo_df["away_stat"] = PeriodInfo_df["away_stat"].astype("Float64")
PeriodInfo_df["total_aces"] = PeriodInfo_df["home_stat"] + PeriodInfo_df["away_stat"]

# Create clean data with only needed columns
PeriodInfo_clean = PeriodInfo_df[["match_id","home_stat","away_stat","total_aces"]]

In [3]:
# Get mean with 2 approaches:
# 1: Do not omit outliers because the data is valid! (Based on Google search)
PeriodInfo_clean.describe()

,match_id,home_stat,away_stat,total_aces
count,1.258200e+04,12582.0,12582.0,12582.0
mean,1.211834e+07,2.800747,2.751629,5.552376
std,5.130522e+04,3.248345,3.213798,5.291285
min,1.199844e+07,0.0,0.0,0.0
25%,1.207829e+07,0.0,0.0,2.0
50%,1.212382e+07,2.0,2.0,4.0
75%,1.215717e+07,4.0,4.0,8.0
max,1.221380e+07,30.0,39.0,51.0


In [4]:
# 2: Omit outliers using IQR value
# Delete outliers of periods

Q1 = PeriodInfo_clean["total_aces"].quantile(0.25)
Q3 = PeriodInfo_clean["total_aces"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df = PeriodInfo_clean[PeriodInfo_clean["total_aces"].isna() |
    ((PeriodInfo_clean["total_aces"] >= lower) &
     ( PeriodInfo_clean["total_aces"]<= upper))
]

df.describe()


,match_id,home_stat,away_stat,total_aces
count,1.210200e+04,12102.0,12102.0,12102.0
mean,1.211886e+07,2.470914,2.41803,4.888944
std,5.115212e+04,2.661544,2.61634,4.081951
min,1.199845e+07,0.0,0.0,0.0
25%,1.207940e+07,0.0,0.0,2.0
50%,1.212393e+07,2.0,2.0,4.0
75%,1.215740e+07,4.0,4.0,7.0
max,1.221380e+07,15.0,16.0,17.0
